### Import librerie


In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Clarke Error Grid Utilities


In [22]:
def _calculate_clarke_zones(ref_values, pred_values):
    """
    Calculate Clarke Error Grid zone assignments for each point
    Args:
        ref_values (numpy.ndarray): Reference glucose values
        pred_values (numpy.ndarray): Predicted glucose values
    Returns:
        list: List of 5 integers representing counts in zones A, B, C, D, E
    """
    zone = [0] * 5

    for i in range(len(ref_values)):
        ref_val = ref_values[i]
        pred_val = pred_values[i]

        # Zone A: Clinically accurate values
        if (ref_val <= 70 and pred_val <= 70) or (
            pred_val <= 1.2 * ref_val and pred_val >= 0.8 * ref_val
        ):
            zone[0] += 1  # Zone A

        # Zone E: Erroneous values (most dangerous)
        elif (ref_val >= 180 and pred_val <= 70) or (ref_val <= 70 and pred_val >= 180):
            zone[4] += 1  # Zone E

        # Zone C: Overcorrection values
        elif ((ref_val >= 70 and ref_val <= 290) and pred_val >= ref_val + 110) or (
            (ref_val >= 130 and ref_val <= 180)
            and (pred_val <= (7 / 5) * ref_val - 182)
        ):
            zone[2] += 1  # Zone C

        # Zone D: Dangerous failure to detect values
        elif (
            (ref_val >= 240 and (pred_val >= 70 and pred_val <= 180))
            or (ref_val <= 175 / 3 and pred_val <= 180 and pred_val >= 70)
            or (
                (ref_val >= 175 / 3 and ref_val <= 70) and pred_val >= (6 / 5) * ref_val
            )
        ):
            zone[3] += 1  # Zone D

        # Zone B: Benign errors
        else:
            zone[1] += 1  # Zone B

    return zone

In [23]:
def _add_clarke_zone_boundaries():
    """Add Clarke Error Grid zone boundary lines to the current plot"""
    zone_line_color = "black"
    zone_line_width = 1.5

    # Zone boundary lines according to Clarke Error Grid specification
    plt.plot([0, 175 / 3], [70, 70], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot(
        [175 / 3, 400 / 1.2],
        [70, 400],
        "-",
        c=zone_line_color,
        linewidth=zone_line_width,
    )
    plt.plot([70, 70], [84, 400], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([0, 70], [180, 180], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([70, 290], [180, 400], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([70, 70], [0, 56], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([70, 400], [56, 320], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([180, 180], [0, 70], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([180, 400], [70, 70], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([240, 240], [70, 180], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([240, 400], [180, 180], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([130, 180], [0, 70], "-", c=zone_line_color, linewidth=zone_line_width)

In [24]:
def _add_clarke_zone_labels():
    """Add zone labels (A, B, C, D, E) to the current plot"""
    zone_font_size = 15
    zone_font_weight = "bold"

    # Zone A
    plt.text(
        30, 15, "A", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone B (appears in two regions)
    plt.text(
        370, 220, "B", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        290, 370, "B", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone C (appears in two regions)
    plt.text(
        160, 370, "C", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        160, 15, "C", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone D (appears in two regions)
    plt.text(
        30, 140, "D", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        370, 90, "D", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone E (appears in two regions)
    plt.text(
        30, 370, "E", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        370, 15, "E", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

In [25]:
def _create_clarke_error_grid_plot(ref_values, pred_values, title_string, save_path):
    """
    Create and save Clarke Error Grid visualization
    Args:
        ref_values (array-like): Valori glicemici di riferimento (mg/dl)
        pred_values (array-like): Valori glicemici inferiti (mg/dl)
        title_string (str): Titolo per il plot
        save_path (str): Path dove salvare il plot. Se None, il plot non viene salvato
    """

    # plt.figure(figsize=(8, 8), dpi=300)

    plt.figure(dpi=300)

    # Plot data points
    plt.scatter(
        ref_values,
        pred_values,
        marker="o",
        color="steelblue",
        s=12,
        # alpha=0.6,
        edgecolors="black",
        linewidth=0.1,
    )

    # Set labels and title

    # plt.title(title_string + " Clarke Error Grid", fontsize=16, fontweight="bold")

    plt.xlabel("Glicemia di riferimento (mg/dL)", fontsize=14)
    plt.ylabel("Glicemia inferita (mg/dL)", fontsize=14)

    # Add perfect prediction line
    plt.plot(
        [0, 400],
        [0, 400],
        ":",
        c="gray",
        linewidth=2,
        alpha=0.7,
        label="Perfect prediction",
    )

    # Add zone boundaries
    _add_clarke_zone_boundaries()

    # Add zone labels
    _add_clarke_zone_labels()

    # Configure plot appearance
    plt.xlim([0, 400])
    plt.ylim([0, 400])
    plt.grid(True, alpha=0.3, linestyle="--")
    plt.gca().set_aspect("equal")
    plt.tight_layout()

    # Save plot
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

In [26]:
def clarke_error_grid_analysis(ref_values, pred_values, title_string, save_path=None):
    """
    Esegui un'analisi tramite la CEG e restituisci le statistiche per ogni zona
    Args:
        ref_values (array-like): Valori glicemici di riferimento (mg/dl)
        pred_values (array-like): Valori glicemici inferiti (mg/dl)
        title_string (str): Titolo per il plot
        save_path (str): Path dove salvare il plot. Se None, il plot non viene salvato
    Returns:
        dict: Dizionario contenente le statistiche per ogni zona:
        - zone_counts: Count dei punti per ciascuna zona (A, B, C, D, E)
        - zone_percentages: Percentuale dei punti in ciascuna zona
        - total_points: Numero totale di punti validi (che ricadono nella griglia)
        - clinically_acceptable: Percentuale nelle zone A+B
        - clinically_dangerous: Percentuale nelle zone D+E
    """
    # Convert to numpy arrays
    ref_values = np.array(ref_values)
    pred_values = np.array(pred_values)

    # Filter out invalid values
    valid_mask = (
        ~np.isnan(ref_values)
        & ~np.isnan(pred_values)
        & (ref_values >= 0)
        & (pred_values >= 0)
        & (ref_values <= 400)
        & (pred_values <= 400)
    )

    ref_values = ref_values[valid_mask]
    pred_values = pred_values[valid_mask]

    # Calculate zone statistics
    zone_counts = _calculate_clarke_zones(ref_values, pred_values)

    total_points = sum(zone_counts)
    zone_percentages = [count / total_points * 100 for count in zone_counts]

    stats = {
        "zone_counts": dict(zip(["A", "B", "C", "D", "E"], zone_counts)),
        "zone_percentages": dict(zip(["A", "B", "C", "D", "E"], zone_percentages)),
        "total_points": total_points,
        "clinically_acceptable": (zone_counts[0] + zone_counts[1]) / total_points * 100,
        "clinically_dangerous": (zone_counts[3] + zone_counts[4]) / total_points * 100,
    }

    # Create plot if save_path is provided
    if save_path:
        _create_clarke_error_grid_plot(ref_values, pred_values, title_string, save_path)
        print(f"Clarke Error Grid saved to: {save_path}")

    return stats

In [27]:
def extract_model_name_from_filename(filename):
    """Estrai il nome del modello dal file di output"""
    # Remove '_output.csv' suffix and convert to uppercase
    if filename.endswith("_output.csv"):
        model_name = filename[:-11].upper()
    else:
        model_name = filename.split(".")[0].upper()

    return model_name

### Main pipeline


In [28]:
# Configura
class Args:
    def __init__(self):
        self.output_path = "outputs/test_set"
        self.plots_path = "plots/test_set"
        self.scores_path = "scores/test_set"


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"  - Output path: {args.output_path}")
print(f"  - Plots path: {args.plots_path}")
print(f"  - Scores path: {args.scores_path}")

Configurazione:
  - Output path: outputs/test_set
  - Plots path: plots/test_set
  - Scores path: scores/test_set


In [29]:
print("=" * 80)
print("CLARKE ERROR GRID ANALYSIS")
print("=" * 80)

# Verifica che sia presente la directory con l'output dei modelli
if not os.path.exists(args.output_path):
    print(f"Output directory not found: {args.output_path}")
else:
    print(f"Output directory found: {args.output_path}")

# Crea le directory di output
os.makedirs(args.plots_path, exist_ok=True)
os.makedirs(args.scores_path, exist_ok=True)

CLARKE ERROR GRID ANALYSIS
Output directory found: outputs/test_set


In [30]:
# Trova i file con gli output dei modelli
output_files = [f for f in os.listdir(args.output_path) if f.endswith("_output.csv")]

if not output_files:
    print(f"No output files found in {args.output_path}")
    print("Looking for files ending with '_output.csv'")

else:
    print(f"Found {len(output_files)} model output files:")
    for file in output_files:
        print(f"  - {file}")

    # Salva i risultati per la tabella di summary finale
    ceg_results = []

    # Processa ogni file di output
    for file in output_files:
        print(f"\n{'='*60}")
        model_name = extract_model_name_from_filename(file)
        print(f"PROCESSING: {model_name}")
        print(f"{'='*60}")

        # Carica i risultati del modello
        file_path = os.path.join(args.output_path, file)
        try:
            model_results = pd.read_csv(file_path)
            print(f"Loaded {len(model_results)} predictions from {file}")
        except Exception as e:
            print(f"Error loading {file}: {str(e)}")
            continue

        # Verifica che siano presenti le colonne specificate
        required_cols = ["target", "y_pred"]
        missing_cols = [
            col for col in required_cols if col not in model_results.columns
        ]
        if missing_cols:
            print(f"Missing required columns in {file}: {missing_cols}")
            continue

        # Estrai i valori glicemici di riferimento e quelli inferiti
        ref_values = model_results["target"].values
        pred_values = model_results["y_pred"].values

        # Genera un CEG plot
        plot_filename = f"ceg_{model_name.lower()}.png"
        plot_path = os.path.join(args.plots_path, plot_filename)

        # Esegui un'analisi dei punti della Clarke Error Grid
        try:
            ceg_stats = clarke_error_grid_analysis(
                ref_values=ref_values,
                pred_values=pred_values,
                title_string=f"{model_name}",
                save_path=plot_path,
            )

            print(f"Clarke Error Grid plot saved: {plot_filename}")

            # Print zone statistics
            print(f"\nClarke Error Grid Zone Statistics:")
            for zone, percentage in ceg_stats["zone_percentages"].items():
                print(f"  Zone {zone}: {percentage:.2f}%")

            print(
                f"  Clinically Acceptable (A+B): {ceg_stats['clinically_acceptable']:.2f}%"
            )
            print(
                f"  Clinically Dangerous (D+E): {ceg_stats['clinically_dangerous']:.2f}%"
            )

            # Store results for summary table
            ceg_results.append(
                {
                    "Model": model_name,
                    "Total_Points": ceg_stats["total_points"],
                    "Zone_A_Pct": f"{ceg_stats['zone_percentages']['A']:.2f}%",
                    "Zone_B_Pct": f"{ceg_stats['zone_percentages']['B']:.2f}%",
                    "Zone_C_Pct": f"{ceg_stats['zone_percentages']['C']:.2f}%",
                    "Zone_D_Pct": f"{ceg_stats['zone_percentages']['D']:.2f}%",
                    "Zone_E_Pct": f"{ceg_stats['zone_percentages']['E']:.2f}%",
                    "Clinically_Acceptable_AandB": f"{ceg_stats['clinically_acceptable']:.2f}%",
                    "Clinically_Dangerous_DandE": f"{ceg_stats['clinically_dangerous']:.2f}%",
                }
            )

        except Exception as e:
            print(f"Error generating Clarke Error Grid for {model_name}: {str(e)}")
            continue

    # Salva la tabella di summary
    if ceg_results:
        print(f"\n{'='*80}")
        print("SAVING CLARKE ERROR GRID SUMMARY")
        print(f"{'='*80}")

        # Costruisci un DataFrame di summary
        ceg_df = pd.DataFrame(ceg_results)

        # Salva in formato CSV
        summary_path = os.path.join(args.scores_path, "ceg_zones_results.csv")
        ceg_df.to_csv(summary_path, index=False)
        print(f"CEG zone statistics saved to: {summary_path}")

        # Display summary table
        print(f"\nClarke Error Grid Zone Summary:")
        print(ceg_df.to_string(index=False))

        # Identifica il miglior modello (quello con percentuale A+B più alta)
        best_model_idx = (
            ceg_df["Clinically_Acceptable_AandB"].str.rstrip("%").astype(float).idxmax()
        )
        best_model = ceg_df.iloc[best_model_idx]
        print(
            f"\nBest Clinical Accuracy: {best_model['Model']} "
            f"({best_model['Clinically_Acceptable_AandB']} in zones A+B)"
        )

    else:
        print("No valid Clarke Error Grid results generated")

    print(f"\n{'='*80}")
    print("CLARKE ERROR GRID ANALYSIS COMPLETED")
    print(f"{'='*80}")
    print(f"Plots saved to: {args.plots_path}")
    print(f"Statistics saved to: {args.scores_path}")

Found 2 model output files:
  - gru_output.csv
  - xgb_output.csv

PROCESSING: GRU
Loaded 18322 predictions from gru_output.csv
Clarke Error Grid saved to: plots/test_set\ceg_gru.png
Clarke Error Grid plot saved: ceg_gru.png

Clarke Error Grid Zone Statistics:
  Zone A: 85.14%
  Zone B: 12.44%
  Zone C: 0.03%
  Zone D: 2.40%
  Zone E: 0.00%
  Clinically Acceptable (A+B): 97.58%
  Clinically Dangerous (D+E): 2.40%

PROCESSING: XGB
Loaded 18322 predictions from xgb_output.csv
Clarke Error Grid saved to: plots/test_set\ceg_xgb.png
Clarke Error Grid plot saved: ceg_xgb.png

Clarke Error Grid Zone Statistics:
  Zone A: 84.72%
  Zone B: 12.40%
  Zone C: 0.03%
  Zone D: 2.85%
  Zone E: 0.00%
  Clinically Acceptable (A+B): 97.12%
  Clinically Dangerous (D+E): 2.85%

SAVING CLARKE ERROR GRID SUMMARY
CEG zone statistics saved to: scores/test_set\ceg_zones_results.csv

Clarke Error Grid Zone Summary:
Model  Total_Points Zone_A_Pct Zone_B_Pct Zone_C_Pct Zone_D_Pct Zone_E_Pct Clinically_Acceptable_